<a href="https://colab.research.google.com/github/thar-26/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thar-26/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

repo = "/content/flyrank-ml-internship"

if not os.path.exists(repo):
    !git clone https://github.com/thar-26/flyrank-ml-internship.git "$repo"

%cd /content/flyrank-ml-internship

print("Current directory:", os.getcwd())
print("CSV exists:", os.path.exists("data/raw/content_refresh_anonymized.csv"))
print("Files in data/raw:", os.listdir("data/raw") if os.path.exists("data/raw") else "data/raw missing")

/content/flyrank-ml-internship
Current directory: /content/flyrank-ml-internship
CSV exists: True
Files in data/raw: ['content_refresh_anonymized.csv']


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane is Refresh / Content Opportunity Scoring, so the output is a ranked review queue rather than an automatic yes/no decision. I use classifier probabilities as ranking scores and evaluate the top of the queue with Precision@20 and Precision@50.

I start with Logistic Regression because it provides a simple and interpretable learned baseline. I then compare it with Random Forest to test whether nonlinear relationships and interactions between page signals improve the ranking enough to justify additional complexity.

The models use only current page-level signals. The proxy label is derived from trend_direction for evaluation, so trend_direction and trend_pct are excluded from the features. IDs are used only for grouping and splitting.

In [2]:
import pandas as pd
import numpy as np
import sklearn

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

SEED = 42

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Evaluation proxy only.
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

features = [
    "impressions_90d",
    "days_since_last_update",
    "content_age_days",
    "avg_position",
    "ctr",
    "engagement_rate",
]

target = "is_declining_label"

print(f"Rows: {len(df):,}")
print(f"Clients: {df['client_id'].nunique()}")
print(f"Features: {len(features)}")
print(f"Proxy target rate: {df[target].mean():.3f}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Random seed: {SEED}")


Rows: 30,000
Clients: 32
Features: 6
Proxy target rate: 0.542
scikit-learn version: 1.6.1
Random seed: 42


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a client-grouped holdout split. All pages from a client belong entirely to either the training set or the test set. This is more honest than a random row split because pages from the same client may share content and performance patterns. A random row split could allow the model to learn client-specific patterns that also appear in the test set.

I use a fixed random seed of 42 so the split and model results are reproducible. The frozen Week-4 baseline and both learned models are evaluated only on the same held-out test rows.

In [3]:
X = df[features].copy()
y = df[target].copy()
groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=SEED,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

X_train = train_df[features]
X_test = test_df[features]

y_train = train_df[target]
y_test = test_df[target]

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

print(f"Training rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")

print(f"Training clients: {len(train_clients)}")
print(f"Test clients: {len(test_clients)}")

print(f"Client overlap: {len(train_clients.intersection(test_clients))}")

print(f"Training target rate: {y_train.mean():.3f}")
print(f"Test target rate: {y_test.mean():.3f}")

assert train_clients.isdisjoint(test_clients)

print("PASS: No client appears in both train and test.")

Training rows: 22,885
Test rows: 7,115
Training clients: 24
Test clients: 8
Client overlap: 0
Training target rate: 0.550
Test target rate: 0.517
PASS: No client appears in both train and test.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I train Logistic Regression and Random Forest models using only the training clients. Predicted probabilities are used as ranking scores because the decision is which pages should be reviewed first.

For a fair comparison, I recompute the frozen Week-4 baseline rule on exactly the same held-out test rows. The baseline rule is unchanged: pages 91-180 days since their last update with at least 500 impressions are eligible, and eligible pages are ranked by impressions.

All methods are compared using Precision@20 and Precision@50 on the same test set. The test base rate is also reported for context.

In [4]:
# --------------------------------------------------
# HELPERS
# --------------------------------------------------

def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    return labels[order[:k]].mean()


# --------------------------------------------------
# FROZEN WEEK-4 BASELINE ON TEST SET
# --------------------------------------------------

baseline_stale_window = (
    (test_df["days_since_last_update"] >= 91) &
    (test_df["days_since_last_update"] <= 180)
)

baseline_visible = (
    test_df["impressions_90d"] >= 500
)

baseline_eligible = (
    baseline_stale_window & baseline_visible
)

baseline_scores = (
    baseline_eligible.astype(int) *
    test_df["impressions_90d"]
)


# --------------------------------------------------
# LOGISTIC REGRESSION
# --------------------------------------------------

logistic_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=SEED,
            ),
        ),
    ]
)

logistic_model.fit(X_train, y_train)

logistic_scores = logistic_model.predict_proba(X_test)[:, 1]


# --------------------------------------------------
# RANDOM FOREST
# --------------------------------------------------

random_forest_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=10,
                random_state=SEED,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_model.fit(X_train, y_train)

random_forest_scores = (
    random_forest_model.predict_proba(X_test)[:, 1]
)


# --------------------------------------------------
# SAME TEST SET + SAME METRICS
# --------------------------------------------------

test_base_rate = y_test.mean()

results = pd.DataFrame(
    {
        "Method": [
            "Test base rate",
            "Frozen Week-4 baseline",
            "Logistic Regression",
            "Random Forest",
        ],
        "Precision@20": [
            test_base_rate,
            precision_at_k(
                baseline_scores,
                y_test,
                20,
            ),
            precision_at_k(
                logistic_scores,
                y_test,
                20,
            ),
            precision_at_k(
                random_forest_scores,
                y_test,
                20,
            ),
        ],
        "Precision@50": [
            test_base_rate,
            precision_at_k(
                baseline_scores,
                y_test,
                50,
            ),
            precision_at_k(
                logistic_scores,
                y_test,
                50,
            ),
            precision_at_k(
                random_forest_scores,
                y_test,
                50,
            ),
        ],
    }
)

print("MODEL VS BASELINE — SAME HELD-OUT CLIENTS")
display(results.round(3))

from pathlib import Path
import json

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

metrics = {
    "split": "client_grouped_holdout",
    "random_seed": SEED,
    "train_clients": len(train_clients),
    "test_clients": len(test_clients),
    "test_base_rate": float(test_base_rate),
    "frozen_baseline_precision_at_20": float(
        precision_at_k(baseline_scores, y_test, 20)
    ),
    "frozen_baseline_precision_at_50": float(
        precision_at_k(baseline_scores, y_test, 50)
    ),
    "logistic_regression_precision_at_20": float(
        precision_at_k(logistic_scores, y_test, 20)
    ),
    "logistic_regression_precision_at_50": float(
        precision_at_k(logistic_scores, y_test, 50)
    ),
    "random_forest_precision_at_20": float(
        precision_at_k(random_forest_scores, y_test, 20)
    ),
    "random_forest_precision_at_50": float(
        precision_at_k(random_forest_scores, y_test, 50)
    ),
}

with open(output_dir / "w05_model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Wrote metrics receipt: work/outputs/w05_model_metrics.json")

MODEL VS BASELINE — SAME HELD-OUT CLIENTS


,Method,Precision@20,Precision@50
0,Test base rate,0.517,0.517
1,Frozen Week-4 baseline,0.200,0.300
2,Logistic Regression,0.700,0.680
3,Random Forest,0.900,0.700


Wrote metrics receipt: work/outputs/w05_model_metrics.json


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The learned models outperform the frozen Week-4 baseline on the same held-out clients. Random Forest achieves the strongest Precision@20 at 0.900, compared with 0.700 for Logistic Regression and 0.200 for the frozen baseline. At Precision@50, Random Forest reaches 0.700 and Logistic Regression reaches 0.680, so the advantage of the more complex model is smaller deeper in the review queue.

I use permutation importance on the held-out test set to inspect which features the Random Forest relies on. I also inspect concrete false positives from the top-ranked recommendations. These cases help show where high model scores do not match the evaluation proxy. The interpretation is directional and limited to this client-grouped holdout; it does not show causal effects or prove that refreshing a recommended page will improve performance.

The permutation-importance results show that impressions_90d is the strongest feature for the Random Forest on this held-out set, followed by avg_position and ctr. These features plausibly relate to the refresh-prioritization task because they describe traffic exposure, search visibility, and click behavior. In contrast, days_since_last_update has slightly negative permutation importance, so staleness does not provide useful standalone predictive information for this fitted model on the held-out clients.

The model-score bands show directional ranking separation. The highest-scoring fifth of pages has an observed declining rate of 0.666, while the lowest-scoring fifth has a rate of 0.408. However, the model still makes confident errors. Three inspected false positives from the top 50 have high model scores but are marked non-declining by the evaluation proxy. All three have zero CTR and zero engagement rate, suggesting that patterns associated with weak engagement can resemble declining pages without always matching the observed proxy outcome.

Overall, Random Forest provides the strongest top-of-queue ranking, with Precision@20 of 0.900 compared with 0.700 for Logistic Regression and 0.200 for the frozen baseline. At Precision@50, the Random Forest advantage over Logistic Regression is small (0.700 versus 0.680), so the additional complexity is most clearly justified when the review capacity is limited to the very top of the queue. These results are limited to one client-grouped holdout split and support decision-making rather than causal claims about whether refreshing a page will improve performance.

In [5]:
# --------------------------------------------------
# PERMUTATION IMPORTANCE
# --------------------------------------------------

perm = permutation_importance(
    random_forest_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=SEED,
    n_jobs=-1,
)

importance_df = pd.DataFrame(
    {
        "feature": features,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }
).sort_values(
    "importance_mean",
    ascending=False,
).reset_index(drop=True)

print("RANDOM FOREST PERMUTATION IMPORTANCE")
display(importance_df.round(4))


# --------------------------------------------------
# TOP-RANKED RANDOM FOREST QUEUE
# --------------------------------------------------

rf_queue = test_df[
    [
        "content_id",
        "impressions_90d",
        "days_since_last_update",
        "content_age_days",
        "avg_position",
        "ctr",
        "engagement_rate",
        "is_declining_label",
    ]
].copy()

rf_queue["model_score"] = random_forest_scores

rf_queue = (
    rf_queue
    .sort_values("model_score", ascending=False)
    .reset_index(drop=True)
)

rf_queue["rank"] = np.arange(1, len(rf_queue) + 1)


# --------------------------------------------------
# ERROR ANALYSIS BY SCORE BAND
# --------------------------------------------------

rf_queue["score_band"] = pd.qcut(
    rf_queue["model_score"],
    q=5,
    duplicates="drop",
)

error_by_band = (
    rf_queue
    .groupby("score_band", observed=True)
    .agg(
        n=("content_id", "size"),
        mean_model_score=("model_score", "mean"),
        observed_declining_rate=("is_declining_label", "mean"),
    )
    .sort_values("mean_model_score", ascending=False)
)

print("\nERROR ANALYSIS BY MODEL-SCORE BAND")
display(error_by_band.round(3))


# --------------------------------------------------
# THREE CONCRETE WRONG CASES
# --------------------------------------------------

wrong_cases = (
    rf_queue[
        (rf_queue["rank"] <= 50) &
        (rf_queue["is_declining_label"] == 0)
    ]
    .head(3)
    .copy()
)

print("\nTHREE CONCRETE WRONG CASES FROM TOP 50")
display(
    wrong_cases[
        [
            "rank",
            "content_id",
            "model_score",
            "impressions_90d",
            "days_since_last_update",
            "content_age_days",
            "avg_position",
            "ctr",
            "engagement_rate",
            "is_declining_label",
        ]
    ].round(3)
)


print("\nWHY THESE CASES ARE HARD")

for _, row in wrong_cases.iterrows():
    print(
        f"Rank {int(row['rank'])}: "
        f"The model gave this page a high score ({row['model_score']:.3f}), "
        f"but the evaluation proxy marks it as non-declining. "
        f"It combines impressions={int(row['impressions_90d']):,}, "
        f"days_since_last_update={int(row['days_since_last_update'])}, "
        f"avg_position={row['avg_position']:.1f}, "
        f"CTR={row['ctr']:.2f}, and "
        f"engagement_rate={row['engagement_rate']:.2f}. "
        f"This combination resembles high-priority pages learned from training clients, "
        f"but it does not match the proxy outcome for this held-out page."
    )


# --------------------------------------------------
# FINAL COMPARISON TABLE
# --------------------------------------------------

print("\nFINAL MODEL VS BASELINE TABLE")
display(results.round(3))

RANDOM FOREST PERMUTATION IMPORTANCE


,feature,importance_mean,importance_std
0,impressions_90d,0.0792,0.0029
1,avg_position,0.0208,0.0021
2,ctr,0.0199,0.0024
3,content_age_days,0.0160,0.0039
4,engagement_rate,0.0010,0.0008
5,days_since_last_update,-0.0095,0.0023



ERROR ANALYSIS BY MODEL-SCORE BAND


,n,mean_model_score,observed_declining_rate
score_band,,,
"(0.743, 0.952]",1423,0.819,0.666
"(0.623, 0.743]",1423,0.682,0.564
"(0.514, 0.623]",1423,0.570,0.517
"(0.381, 0.514]",1423,0.450,0.428
"(0.0019399999999999999, 0.381]",1423,0.260,0.408



THREE CONCRETE WRONG CASES FROM TOP 50


,rank,content_id,model_score,impressions_90d,days_since_last_update,content_age_days,avg_position,ctr,engagement_rate,is_declining_label
1,2,content_91c02d2830b8,0.952,720,20,147,19.4,0.0,0.0,0
15,16,content_68742c70d0fa,0.921,660,20,147,7.4,0.0,0.0,0
20,21,content_fb79d33c4d8a,0.920,978,20,147,23.9,0.0,0.0,0



WHY THESE CASES ARE HARD
Rank 2: The model gave this page a high score (0.952), but the evaluation proxy marks it as non-declining. It combines impressions=720, days_since_last_update=20, avg_position=19.4, CTR=0.00, and engagement_rate=0.00. This combination resembles high-priority pages learned from training clients, but it does not match the proxy outcome for this held-out page.
Rank 16: The model gave this page a high score (0.921), but the evaluation proxy marks it as non-declining. It combines impressions=660, days_since_last_update=20, avg_position=7.4, CTR=0.00, and engagement_rate=0.00. This combination resembles high-priority pages learned from training clients, but it does not match the proxy outcome for this held-out page.
Rank 21: The model gave this page a high score (0.920), but the evaluation proxy marks it as non-declining. It combines impressions=978, days_since_last_update=20, avg_position=23.9, CTR=0.00, and engagement_rate=0.00. This combination resembles high-pri

,Method,Precision@20,Precision@50
0,Test base rate,0.517,0.517
1,Frozen Week-4 baseline,0.200,0.300
2,Logistic Regression,0.700,0.680
3,Random Forest,0.900,0.700


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.